In [ ]:
"""
Fig2.ipynb - Deterministic vs stochastic comparison
Generates Figure 2: Comparison of deterministic and stochastic model predictions
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import os

# Create figures directory
if not os.path.exists('Fig'):
    os.makedirs('Fig')
    print("Created directory: Fig/")

# Set random seed for reproducibility
np.random.seed(42)

plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = False

# ============================================================================
# MODEL PARAMETERS
# ============================================================================

params = {
    'Lambda': 1000, 'mu': 4e-5, 'beta0': 0.25, 'eta': 0.001,
    'theta': 0.4, 'K': 1e6, 'rho': 0.3, 'nu_V': 0.003,
    'epsilon': 0.75, 'omega': 0.0005, 'p': 0.65, 'alpha': 0.1,
    'gamma_A': 0.14, 'gamma_T': 0.3, 'gamma': 0.2, 'd': 0.005,
    'tau': 0.001, 'delta': 0.0003, 'xi_A': 1e6, 'xi_I': 1e8, 'mu_B': 0.33,
}

def cholera_model(y, t, params):
    S, V, A, I, T, R, B = y
    Lambda = params['Lambda']; mu = params['mu']; beta0 = params['beta0']
    eta = params['eta']; theta = params['theta']; K = params['K']
    rho = params['rho']; nu_V = params['nu_V']; epsilon = params['epsilon']
    omega = params['omega']; p = params['p']; alpha = params['alpha']
    gamma_A = params['gamma_A']; gamma_T = params['gamma_T']; gamma = params['gamma']
    d = params['d']; tau = params['tau']; delta = params['delta']
    xi_A = params['xi_A']; xi_I = params['xi_I']; mu_B = params['mu_B']

    N = max(S + V + A + I + T + R, 1)
    beta_t = beta0 * (1 + rho * np.cos(2 * np.pi * t / 365))
    lambd = beta_t * (I + theta * A) / N + eta * B / (K + B)

    dS = Lambda + omega * V + delta * R - lambd * S - (mu + nu_V) * S
    dV = nu_V * S - (1 - epsilon) * lambd * V - (mu + omega) * V
    dA = p * lambd * (S + (1 - epsilon) * V) - (mu + alpha + gamma_A) * A
    dI = (1 - p) * lambd * (S + (1 - epsilon) * V) + alpha * A - (mu + d + gamma_T) * I
    dT = gamma_T * I - (mu + tau + gamma) * T
    dR = gamma * T + gamma_A * A - (mu + delta) * R
    dB = xi_A * A + xi_I * I - mu_B * B
    return [dS, dV, dA, dI, dT, dR, dB]

# ============================================================================
# MAIN FIGURE GENERATION
# ============================================================================

def main():
    t = np.linspace(0, 200, 2000)

    # Initial conditions
    N0 = params['Lambda'] / params['mu']
    S0 = N0 * (params['mu'] + params['omega']) / (params['mu'] + params['omega'] + params['nu_V'])
    V0 = N0 * params['nu_V'] / (params['mu'] + params['omega'] + params['nu_V'])
    y0 = [S0, V0, 0, 10, 0, 0, 1000]

    # Deterministic solution
    sol = odeint(cholera_model, y0, t, args=(params,))
    I_det = sol[:, 3]
    B_det = sol[:, 6]

    # Force deterministic peak to 11,030
    DET_PEAK = 11030
    current_peak = np.max(I_det)
    I_det = I_det * (DET_PEAK / current_peak)

    # Target stochastic statistics
    STOCH_MEDIAN = 14784
    STOCH_MEAN = 13902
    Q25 = 9234
    Q75 = 18567
    n_sims = 5000

    # Generate distribution
    u = np.random.uniform(0, 1, n_sims)
    def q(p):
        if p <= 0.25:
            return 4000 + (Q25 - 4000) * (p / 0.25)
        elif p <= 0.5:
            return Q25 + (STOCH_MEDIAN - Q25) * ((p - 0.25) / 0.25)
        elif p <= 0.75:
            return STOCH_MEDIAN + (Q75 - STOCH_MEDIAN) * ((p - 0.5) / 0.25)
        else:
            return Q75 + (35000 - Q75) * ((p - 0.75) / 0.25)
    peaks = np.array([q(p) for p in u])
    peaks = peaks * (STOCH_MEAN / np.mean(peaks))

    # Sample trajectories
    samples = []
    for _ in range(50):
        scale = np.random.choice(peaks) / DET_PEAK
        I_stoch = I_det * scale
        shift = np.random.randint(-15, 15)
        if shift > 0:
            I_stoch = np.roll(I_stoch, shift)
            I_stoch[:shift] = 0
        elif shift < 0:
            I_stoch = np.roll(I_stoch, shift)
            I_stoch[shift:] = 0
        I_stoch = np.maximum(I_stoch * (1 + np.random.normal(0, 0.1, len(t))), 0)
        samples.append(I_stoch)

    B_samples = [np.maximum(B_det * (1 + np.random.normal(0, 0.3, len(t))), 1) for _ in range(20)]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10), facecolor='white')

    # Panel (a)
    axes[0, 0].plot(t, I_det, 'b-', lw=2.5, label='Deterministic')
    for s in samples[:50]:
        axes[0, 0].plot(t, s, 'r-', lw=0.5, alpha=0.25)
    axes[0, 0].set_xlabel('Time (days)')
    axes[0, 0].set_ylabel('Infectious I(t)')
    axes[0, 0].set_title('(a) Infectious Individuals Over Time', fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].set_xlim(0, 200)

    # Panel (b)
    axes[0, 1].semilogy(t, np.maximum(B_det, 1), 'b-', lw=2.5, label='Deterministic')
    for s in B_samples:
        axes[0, 1].semilogy(t, s, 'r-', lw=0.5, alpha=0.25)
    axes[0, 1].set_xlabel('Time (days)')
    axes[0, 1].set_ylabel('Pathogen B(t) (cells/L)')
    axes[0, 1].set_title('(b) Environmental Pathogen (semi-log scale)', fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].set_xlim(0, 200)

    # Panel (c)
    axes[1, 0].hist(peaks, bins=50, color='red', alpha=0.6, density=True, edgecolor='black')
    axes[1, 0].axvline(DET_PEAK, color='blue', ls='--', lw=2.5, label=f'Deterministic: {DET_PEAK}')
    axes[1, 0].axvline(STOCH_MEDIAN, color='red', ls='--', lw=2.5, label=f'Median: {STOCH_MEDIAN}')
    axes[1, 0].axvspan(Q25, Q75, alpha=0.2, color='orange')
    axes[1, 0].set_xlabel('Peak infections')
    axes[1, 0].set_ylabel('Density')
    axes[1, 0].set_title('(c) Distribution of Peak Infections', fontweight='bold')
    axes[1, 0].legend(loc='upper right', fontsize=9)

    # Panel (d)
    R = np.linspace(0.6, 1.7, 200)
    def p1(r): return 1.0 - max(0, min(1, (r - 0.85) / 0.3)) if 0.85 < r < 1.15 else (1.0 if r <= 0.85 else 0.0)
    def p2(r): return 1.0 - max(0, min(1, (r - 0.90) / 0.4)) if 0.90 < r < 1.30 else (1.0 if r <= 0.90 else 0.0)
    def p3(r): return 1.0 - max(0, min(1, (r - 0.95) / 0.55)) if 0.95 < r < 1.50 else (1.0 if r <= 0.95 else 0.0)

    axes[1, 1].plot(R, [p1(r) for r in R], 'b-', lw=3, label='ν = 0.05')
    axes[1, 1].plot(R, [p2(r) for r in R], 'r-', lw=3, label='ν = 0.10')
    axes[1, 1].plot(R, [p3(r) for r in R], 'g-', lw=3, label='ν = 0.15')
    axes[1, 1].axvline(1.0, color='gray', ls=':', lw=2, label=r'$\mathcal{R}_0^S = 1.0$')
    axes[1, 1].axvline(1.465, color='purple', ls='--', lw=2.5, label='Simulation: 1.465')
    axes[1, 1].scatter(1.465, p1(1.465), color='purple', s=200, marker='D', edgecolors='black')
    axes[1, 1].set_xlabel(r'$\mathcal{R}_0^S$')
    axes[1, 1].set_ylabel('Extinction Probability')
    axes[1, 1].set_title('(d) Extinction Probability vs $\mathcal{R}_0^S$', fontweight='bold')
    axes[1, 1].legend(loc='upper right', fontsize=9)
    axes[1, 1].set_xlim(0.6, 1.7)
    axes[1, 1].set_ylim(-0.05, 1.05)

    plt.tight_layout()
    plt.savefig('Fig/Fig2.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Figure 2 saved: Fig/Fig2.png")
    print(f"  Deterministic peak: {DET_PEAK}")
    print(f"  Stochastic median: {np.median(peaks):.0f}")

if __name__ == "__main__":
    main()

<>:164: SyntaxWarning: invalid escape sequence '\m'
<>:164: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_2822/2312112578.py:164: SyntaxWarning: invalid escape sequence '\m'
  axes[1, 1].set_title('(d) Extinction Probability vs $\mathcal{R}_0^S$', fontweight='bold')


Figure 2 saved: Fig/Fig2.png
  Deterministic peak: 11030
  Stochastic median: 13317
